# 🛒 Depuración de Datos — Supermercado de Verduras
## Preparación de Tablas para Modelo Estrella en Power BI

---

| | |
|---|---|
| **Autor** | Diego Encalada |
| **Cargo** | Analista BI Jr. – Ambiensa S.A. |
| **Dataset** | Supermarket Sales Data – Kaggle (yapwh1208) |
| **Fuente** | https://www.kaggle.com/datasets/yapwh1208/supermarket-sales-data |
| **Descripción** | Sales Data of Vegetables in Supermarket (China) |
| **Período** | Julio 2020 – Junio 2023 |
| **Herramientas** | Python · Pandas |
| **Moneda original** | RMB (Yuan Chino) → convertido a USD (1 RMB = 0.14 USD) |

---

## 🎯 Objetivo
Depurar y limpiar los 4 archivos del dataset original para exportarlos como tablas limpias y estructuradas, listas para construir el **Modelo Estrella en Power BI**.

> ⚠️ **Nota:** Este notebook **NO une las tablas**. Cada archivo se depura de forma independiente. Las relaciones entre tablas se crean directamente en Power BI.

> 💱 **Conversión de moneda:** Los datos originales están en RMB. Se agrega columna en USD con tipo de cambio aproximado de 1 RMB = 0.14 USD.

> 📌 **Formato de exportación:** Separador `;` y decimal `,` (formato español). Power BI en español los leerá directamente sin configuración adicional.

---

## 📁 Archivos de entrada
| Archivo | Descripción | Filas |
|---|---|---|
| `annex1.csv` | Catálogo de productos y categorías | 251 |
| `annex2.csv` | Transacciones de ventas individuales | 878,503 |
| `annex3.csv` | Precios mayoristas por producto y fecha | 55,982 |
| `annex4.csv` | Tasa de pérdida (merma) por producto | 251 |

## 📤 Archivos de salida para Power BI
| Archivo | Tipo | Descripción |
|---|---|---|
| `fact_ventas.csv` | **Tabla de Hechos** | Transacciones depuradas con precios en RMB y USD |
| `dim_producto.csv` | **Dimensión** | Catálogo de productos limpio |
| `dim_fecha.csv` | **Dimensión** | Calendario generado con atributos temporales |
| `dim_precio_mayorista.csv` | **Dimensión** | Precios de compra en RMB y USD |
| `dim_perdidas.csv` | **Dimensión** | Tasa de pérdida por producto |

---

## ⭐ Modelo Estrella en Power BI
```
                    dim_fecha
                        | Date (1)
fact_ventas[Date*] ─────┘
fact_ventas[Item Code*] ──► dim_producto[Item_Code(1)]
                                  │
                          dim_perdidas[Item_Code*]
                          dim_precio_mayorista → LOOKUPVALUE en DAX
```

### Relaciones a crear en Power BI
| Desde (*) | Campo | Hacia (1) | Campo |
|---|---|---|---|
| fact_ventas | Item Code | dim_producto | Item_Code |
| fact_ventas | Date | dim_fecha | Date |
| dim_perdidas | Item_Code | dim_producto | Item_Code |


---
## 1. 📦 Importación de Librerías y Configuración


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Tipo de cambio RMB a USD (aproximado)
TIPO_CAMBIO_USD = 0.14

# Formato de exportación para Power BI en español
# sep=';'     → punto y coma como separador de columnas
# decimal=',' → coma como separador decimal
# Así Power BI en Windows español lo lee sin configuración extra
SEP     = ';'
DECIMAL = ','

print("✅ Librerías cargadas")
print(f"   Pandas: {pd.__version__}")
print(f"   Tipo de cambio: 1 RMB = {TIPO_CAMBIO_USD} USD")
print(f"   Formato export: separador='{SEP}' decimal='{DECIMAL}'")

---
## 2. 📥 Carga de Datos Originales

> ⚠️ `annex4.csv` usa `encoding='utf-8-sig'` por el BOM al inicio del archivo.


In [ ]:
productos   = pd.read_csv('annex1.csv')
ventas      = pd.read_csv('annex2.csv')
precios_may = pd.read_csv('annex3.csv')
perdidas    = pd.read_csv('annex4.csv', encoding='utf-8-sig')

print("=" * 55)
print(f"{'Archivo':<25} {'Filas':>10} {'Columnas':>10}")
print("-" * 55)
print(f"{'annex1 (Productos)':<25} {productos.shape[0]:>10,} {productos.shape[1]:>10}")
print(f"{'annex2 (Ventas)':<25} {ventas.shape[0]:>10,} {ventas.shape[1]:>10}")
print(f"{'annex3 (Precios May.)':<25} {precios_may.shape[0]:>10,} {precios_may.shape[1]:>10}")
print(f"{'annex4 (Pérdidas)':<25} {perdidas.shape[0]:>10,} {perdidas.shape[1]:>10}")
print("=" * 55)

---
## 3. 🔍 Exploración Inicial


In [ ]:
for nombre, df in [('PRODUCTOS', productos), ('VENTAS', ventas),
                    ('PRECIOS MAY.', precios_may), ('PÉRDIDAS', perdidas)]:
    print(f"\n{'='*55}")
    print(f" {nombre}")
    print(f"{'='*55}")
    print(f"Columnas: {list(df.columns)}")
    print(f"Tipos:\n{df.dtypes}")
    print(f"\nNulos:\n{df.isnull().sum()}")

In [ ]:
print("MUESTRA DE VENTAS (5 filas):")
ventas.head()

In [ ]:
print("DISTRIBUCIÓN SALE / RETURN:")
print(ventas['Sale or Return'].value_counts())
print(f"\n% devoluciones: {(ventas['Sale or Return']=='return').mean()*100:.2f}%")
print(f"Valores negativos en Quantity: {(ventas['Quantity Sold (kilo)'] < 0).sum():,}")
print(f"Rango Quantity: {ventas['Quantity Sold (kilo)'].min()} → {ventas['Quantity Sold (kilo)'].max()}")
print(f"Rango Price:    {ventas['Unit Selling Price (RMB/kg)'].min()} → {ventas['Unit Selling Price (RMB/kg)'].max()} RMB/kg")

---
## 4. 🧹 Depuración — `fact_ventas`

**Transformaciones aplicadas:**
- Convertir `Date` a datetime y extraer componentes temporales
- Filtrar devoluciones (solo `sale`) y cantidades negativas
- Calcular `Total_Rev_RMB = Quantity_Kg × Unit_Price_RMB`
- Convertir precios a USD con tipo de cambio 0.14
- Convertir `Discount` a flag numérico (0/1)

**Nota del dataset:**
Cada fila es una venta individual de verduras por peso (kg). El precio está en RMB/kg y la cantidad en kilogramos.


In [ ]:
# Fechas y componentes temporales
ventas['Date']       = pd.to_datetime(ventas['Date'])
ventas['Year']       = ventas['Date'].dt.year
ventas['Quarter']    = ventas['Date'].dt.quarter
ventas['Month']      = ventas['Date'].dt.month
ventas['Month_Name'] = ventas['Date'].dt.strftime('%B')
ventas['Week']       = ventas['Date'].dt.isocalendar().week.astype(int)
ventas['Day_Name']   = ventas['Date'].dt.strftime('%A')
ventas['Hour']       = pd.to_datetime(ventas['Time'], format='mixed').dt.hour
print("✅ Componentes temporales extraídos")

# Filtros de calidad
antes = ventas.shape[0]
ventas = ventas[ventas['Sale or Return'] == 'sale'].copy()
ventas = ventas[ventas['Quantity Sold (kilo)'] > 0].copy()
print(f"✅ Filtros aplicados: {antes - ventas.shape[0]:,} registros eliminados")
print(f"   Registros finales: {ventas.shape[0]:,}")

In [ ]:
# Métricas en RMB
ventas['Quantity_Kg']    = ventas['Quantity Sold (kilo)'].round(4)
ventas['Unit_Price_RMB'] = ventas['Unit Selling Price (RMB/kg)'].round(4)
ventas['Total_Rev_RMB']  = (ventas['Quantity_Kg'] * ventas['Unit_Price_RMB']).round(4)
print(f"✅ Métricas RMB: Total = {ventas['Total_Rev_RMB'].sum():,.2f} RMB")

# Conversión a USD
ventas['Unit_Price_USD'] = (ventas['Unit_Price_RMB'] * TIPO_CAMBIO_USD).round(4)
ventas['Total_Rev_USD']  = (ventas['Total_Rev_RMB']  * TIPO_CAMBIO_USD).round(4)
print(f"✅ Conversión USD:  Total = ${ventas['Total_Rev_USD'].sum():,.2f} USD")

# Discount flag
ventas['Discount_Flag'] = (ventas['Discount (Yes/No)'] == 'Yes').astype(int)
print(f"✅ Discount_Flag: {ventas['Discount_Flag'].mean()*100:.1f}% con descuento")

# Selección final
fact_ventas = ventas[[
    'Date', 'Item Code', 'Hour',
    'Quantity_Kg', 'Unit_Price_RMB', 'Unit_Price_USD',
    'Total_Rev_RMB', 'Total_Rev_USD', 'Discount_Flag',
    'Year', 'Quarter', 'Month', 'Month_Name', 'Week', 'Day_Name'
]].copy()

print(f"\n📊 FACT_VENTAS: {fact_ventas.shape[0]:,} filas x {fact_ventas.shape[1]} columnas")
fact_ventas.head(3)

---
## 5. 🧹 Depuración — `dim_producto`

Catálogo de 251 productos con 6 categorías de verduras.


In [ ]:
productos.columns = productos.columns.str.strip()

dim_producto = productos.rename(columns={
    'Item Code'     : 'Item_Code',
    'Item Name'     : 'Item_Name',
    'Category Code' : 'Category_Code',
    'Category Name' : 'Category_Name'
}).copy()

print(f"📦 DIM_PRODUCTO: {dim_producto.shape[0]} filas x {dim_producto.shape[1]} columnas")
print(f"   Categorías: {list(dim_producto['Category_Name'].unique())}")
print(f"   Nulos: {dim_producto.isnull().sum().sum()}")
dim_producto.head(3)

---
## 6. 🧹 Generación — `dim_fecha`

Tabla calendario continua sin huecos. Necesaria para comparativas temporales en Power BI.


In [ ]:
fechas = pd.date_range(
    start=fact_ventas['Date'].min(),
    end=fact_ventas['Date'].max(),
    freq='D'
)

dim_fecha = pd.DataFrame({
    'Date'       : fechas,
    'Year'       : fechas.year,
    'Quarter'    : fechas.quarter,
    'Month'      : fechas.month,
    'Month_Name' : fechas.strftime('%B'),
    'Week'       : fechas.isocalendar().week.astype(int),
    'Day'        : fechas.day,
    'Day_Name'   : fechas.strftime('%A'),
    'Is_Weekend' : (fechas.weekday >= 5)
})

print(f"📅 DIM_FECHA: {dim_fecha.shape[0]} días continuos")
print(f"   Período: {fechas[0].date()} → {fechas[-1].date()}")
dim_fecha.head(3)

---
## 7. 🧹 Depuración — `dim_precio_mayorista`

Precios de compra por producto y fecha en RMB y USD.

> ⚠️ Esta tabla se usa con **LOOKUPVALUE** en DAX, no con relación directa en Power BI, porque tiene granularidad de precio por producto por día (muchos a muchos).


In [ ]:
precios_may['Date'] = pd.to_datetime(precios_may['Date'])

dim_precio_mayorista = precios_may.rename(columns={
    'Item Code'                : 'Item_Code',
    'Wholesale Price (RMB/kg)' : 'Wholesale_Price_RMB'
}).copy()

dim_precio_mayorista['Wholesale_Price_USD'] = (
    dim_precio_mayorista['Wholesale_Price_RMB'] * TIPO_CAMBIO_USD
).round(4)

print(f"💲 DIM_PRECIO_MAYORISTA: {dim_precio_mayorista.shape[0]:,} filas")
print(f"   Precio promedio RMB: {dim_precio_mayorista['Wholesale_Price_RMB'].mean():.2f}")
print(f"   Precio promedio USD: ${dim_precio_mayorista['Wholesale_Price_USD'].mean():.2f}")
dim_precio_mayorista.head(3)

---
## 8. 🧹 Depuración — `dim_perdidas`

Tasa de merma por producto. Se relaciona con `dim_producto` mediante `Item_Code`.


In [ ]:
perdidas.columns = perdidas.columns.str.strip()

dim_perdidas = perdidas.rename(columns={
    'Item Code'     : 'Item_Code',
    'Item Name'     : 'Item_Name',
    'Loss Rate (%)' : 'Loss_Rate_Pct'
}).copy()

dim_perdidas['Loss_Rate_Pct'] = pd.to_numeric(
    dim_perdidas['Loss_Rate_Pct'].astype(str).str.strip(), errors='coerce'
).round(2)

print(f"📉 DIM_PERDIDAS: {dim_perdidas.shape[0]} filas")
print(f"   Pérdida promedio: {dim_perdidas['Loss_Rate_Pct'].mean():.2f}%")
print(f"   Pérdida máxima:   {dim_perdidas['Loss_Rate_Pct'].max():.2f}%")
dim_perdidas.head(3)

---
## 9. ✅ Validación Final


In [ ]:
print("VALIDACIÓN DE NULOS")
print("=" * 50)
for nombre, tabla in [('fact_ventas', fact_ventas), ('dim_producto', dim_producto),
                       ('dim_fecha', dim_fecha), ('dim_precio_mayorista', dim_precio_mayorista),
                       ('dim_perdidas', dim_perdidas)]:
    nulos = tabla.isnull().sum().sum()
    print(f"  {nombre:<30} {'✅ Sin nulos' if nulos == 0 else f'⚠️ {nulos} nulos'}")

print("\nVALIDACIÓN REFERENCIAL")
print("=" * 50)
codes_fact = set(fact_ventas['Item Code'].unique())
codes_prod = set(dim_producto['Item_Code'].unique())
huerfanos  = codes_fact - codes_prod
print(f"  fact_ventas → dim_producto: {'✅ OK' if not huerfanos else f'⚠️ {len(huerfanos)} sin match'}")

print(f"\nKPIs FINALES:")
print(f"  Período:      {fact_ventas['Date'].min().date()} → {fact_ventas['Date'].max().date()}")
print(f"  Transacciones:{fact_ventas.shape[0]:,}")
print(f"  Ingresos RMB: {fact_ventas['Total_Rev_RMB'].sum():,.2f}")
print(f"  Ingresos USD: ${fact_ventas['Total_Rev_USD'].sum():,.2f}")
print(f"  Kg vendidos:  {fact_ventas['Quantity_Kg'].sum():,.2f}")
print(f"  % descuento:  {fact_ventas['Discount_Flag'].mean()*100:.1f}%")

---
## 10. 💾 Exportación para Power BI

Formato español: separador `;` y decimal `,`
Power BI en Windows español los lee directamente sin configuración extra.


In [ ]:
tablas = {
    'fact_ventas.csv'          : fact_ventas,
    'dim_producto.csv'         : dim_producto,
    'dim_fecha.csv'            : dim_fecha,
    'dim_precio_mayorista.csv' : dim_precio_mayorista,
    'dim_perdidas.csv'         : dim_perdidas
}

print("EXPORTANDO TABLAS...")
print("=" * 60)
for nombre, tabla in tablas.items():
    tabla.to_csv(nombre, index=False, encoding='utf-8-sig', sep=SEP, decimal=DECIMAL)
    print(f"✅ {nombre:<35} {tabla.shape[0]:>8,} filas")

print("=" * 60)
print(f"\n💱 Tipo de cambio: 1 RMB = {TIPO_CAMBIO_USD} USD")
print(f"💵 Total USD: ${fact_ventas['Total_Rev_USD'].sum():,.2f}")
print(f"💰 Total RMB: {fact_ventas['Total_Rev_RMB'].sum():,.2f}")
print(f"\n🎯 PASOS EN POWER BI:")
print("   1. Obtener datos → Texto/CSV → cargar los 5 archivos")
print("   2. Vista Modelo → crear relaciones:")
print("      fact_ventas[Item Code] → dim_producto[Item_Code]  (*→1)")
print("      fact_ventas[Date]      → dim_fecha[Date]          (*→1)")
print("      dim_perdidas[Item_Code]→ dim_producto[Item_Code]  (*→1)")
print("   3. dim_precio_mayorista se usa con LOOKUPVALUE en DAX")

---
## 11. 📐 Medidas DAX para Power BI

### KPIs Principales
```dax
Total Ingresos USD = SUM(fact_ventas[Total_Rev_USD])
Total Ingresos RMB = SUM(fact_ventas[Total_Rev_RMB])
Total Kg Vendidos = SUM(fact_ventas[Quantity_Kg])
Total Transacciones = COUNTROWS(fact_ventas)
Precio Promedio USD = AVERAGE(fact_ventas[Unit_Price_USD])
% Con Descuento =
    DIVIDE(
        COUNTROWS(FILTER(fact_ventas, fact_ventas[Discount_Flag] = 1)),
        COUNTROWS(fact_ventas)
    ) * 100
```

### Análisis Temporal
```dax
Ingresos Mes Anterior =
    CALCULATE([Total Ingresos USD], DATEADD(dim_fecha[Date], -1, MONTH))

Crecimiento MoM % =
    DIVIDE([Total Ingresos USD] - [Ingresos Mes Anterior], [Ingresos Mes Anterior]) * 100

Ingresos Año Anterior =
    CALCULATE([Total Ingresos USD], DATEADD(dim_fecha[Date], -1, YEAR))

Crecimiento YoY % =
    DIVIDE([Total Ingresos USD] - [Ingresos Año Anterior], [Ingresos Año Anterior]) * 100
```

### Rentabilidad
```dax
Costo Total USD =
    SUMX(
        fact_ventas,
        fact_ventas[Quantity_Kg] *
        LOOKUPVALUE(
            dim_precio_mayorista[Wholesale_Price_USD],
            dim_precio_mayorista[Item_Code], fact_ventas[Item Code],
            dim_precio_mayorista[Date], fact_ventas[Date]
        )
    )

Margen Bruto USD = [Total Ingresos USD] - [Costo Total USD]
Margen % = DIVIDE([Margen Bruto USD], [Total Ingresos USD]) * 100
```

### Pérdidas
```dax
Kg Perdidos Estimados =
    SUMX(
        fact_ventas,
        fact_ventas[Quantity_Kg] *
        LOOKUPVALUE(
            dim_perdidas[Loss_Rate_Pct],
            dim_perdidas[Item_Code], fact_ventas[Item Code]
        ) / 100
    )
```

---

## 📝 Resumen

| | |
|---|---|
| **Dataset** | Ventas de verduras en supermercado chino (Kaggle) |
| **Registros originales** | 878,503 |
| **Tras depuración** | 878,042 |
| **Tablas exportadas** | 5 tablas para modelo estrella |
| **Moneda** | RMB + USD (×0.14) |
| **Total Ingresos** | RMB 3,372,023 / USD 472,083 |
| **Formato export** | CSV separador `;` decimal `,` |

> 🚀 **GitHub:** [link a tu repositorio]
> 👤 **Autor:** Diego Encalada | Analista BI Jr. | Python + Power BI
